# VLM-DENTAL — Stage 2: Group Relative Policy Optimization (GRPO)

This notebook optimizes **Qwen/Qwen3.5-9B** using Stage 2 Group Relative Policy Optimization (GRPO) reinforcement learning (§17) against verifiable clinical ground truth.

### Core Architectural & Clinical Invariants:
- **Curriculum Reference Integration**: Initializes the frozen reference policy directly from the Stage 1 SFT checkpoint corresponding to the curriculum stage:
  - Stage 1a: `dentex_alone` $\rightarrow$ `qwen3_5_9b_sft_{track}_dentex_alone`
  - Stage 1b: `dentex_tufts_overlap` $\rightarrow$ `qwen3_5_9b_sft_{track}_dentex_tufts_overlap`
  - Stage 1c: `multicohort_all` $\rightarrow$ `qwen3_5_9b_sft_{track}_multicohort_all`
- **[G3] Dual-LoRA In-Memory Toggle**: Hosts both the frozen SFT reference (`"reference"`) and trainable RL policy (`"grpo_policy"`) on a single base model in memory, eliminating redundant VRAM overhead.
- **[G2] Batched Rollout Generation**: Batches all $K$ candidate trajectories into a single forward-pass rollout to eliminate sequential decode latency bottlenecks.
- **Flexible Group Size ($K \in \{1, 2, 4, 8, 16\}$)**: Full support for single runs or automated parameter sweeps with EMA baseline fallback for $K=1$ and tie-breaking for $K=2$.
- **Rule 13 Multi-Finding Bipartite Evaluation**: Complete Hungarian set-level matching across all $1$ to $7$ findings per panoramic radiograph (zero `.iloc[0]` truncation).
- **Cloud TPU v5e-8 Hardware Optimization**: Native 8-way distributed data-parallel execution (xmp.spawn) leveraging 128 GB total HBM on Google Cloud TPU / Kaggle.
- **Unified Hugging Face Hub Models Repository**: Checkpoints are stored in `Reza-Nadimi/vlm-dental-models` under structured folders (`grpo/qwen3_5_9b_grpo_{track}_k{group_size}_{stage}/`), synced every 25 steps for seamless cross-account session resume.

## 1. Platform Detection & Shallow Repository Clone

Detects runtime environment (Kaggle vs Colab vs Local) and performs a shallow clone (`--depth 1`) to eliminate Git history download overhead.

In [ ]:
import os
import sys
from pathlib import Path

# 1. Detect platform environment
IS_KAGGLE = os.path.exists("/kaggle") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in os.environ
PLATFORM_NAME = "Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else "Local PC / Server")
print(f"[PLATFORM] Detected Runtime Environment: {PLATFORM_NAME}")

# 2. Shallow clone repository (--depth 1) to conserve bandwidth, disk space, and time
REPO_NAME = "VLM-DENTAL"
REPO_URL = "https://github.com/rezaxr14/VLM-DENTAL.git"

if not os.path.exists(REPO_NAME) and not os.path.exists("dental_agent"):
    print(f"[CLONE] Performing shallow clone (--depth 1) of {REPO_URL}...")
    !git clone --depth 1 {REPO_URL}
    %cd {REPO_NAME}
elif os.path.exists(REPO_NAME):
    %cd {REPO_NAME}
    print(f"[WORKSPACE] Switched directory to {os.getcwd()}")
else:
    print(f"[WORKSPACE] Already inside repository root: {os.getcwd()}")

## 2. Hardened Dependency Installation & Environment Setup

Installs project dependencies before calling any hardware-specific libraries. Configures `PJRT_DEVICE=TPU` for Kaggle TPU v5e-8.

In [ ]:
# Install VLM-DENTAL in editable mode and essential PEFT / training packages
# Note: vLLM is completely purged to avoid CUDA binary incompatibilities on TPU
!pip install -q -e .
!pip install -q peft trl datasets accelerate huggingface_hub ultralytics python-dotenv qwen-vl-utils tabulate pytest

## 3. Hardware & Cloud TPU v5e-8 Topology Detection

Initializes PyTorch/XLA on TPU v5e-8 (reporting 8-way Distributed Data-Parallel (xmp.spawn) device count and chip topology) or reports CUDA GPUs.

In [ ]:
import os
import torch

IS_TPU = False
DEVICE_STR = "cpu"

# Detect TPU availability without initializing the PJRT hardware client in the
# notebook kernel. Calling xm.xla_device() in the notebook kernel locks /dev/vfio/*
# hardware devices, which prevents child training scripts (!python scripts/train_sft.py)
# from spawning multi-core workers with 'open(/dev/vfio/1): Device or resource busy'.
if os.path.exists("/dev/vfio") or os.environ.get("PJRT_DEVICE") == "TPU" or "COLAB_TPU_ADDR" in os.environ:
    try:
        import torch_xla
        IS_TPU = True
        DEVICE_STR = "Cloud TPU (8 cores / chips via PyTorch/XLA)"
        print(f"[HARDWARE] SUCCESS: Detected {DEVICE_STR}")
        print(f"[HARDWARE] 8-Way PyTorch/XLA FSDP (xmp.spawn) ready across 128 GB total HBM (2.3 GB / core).")
    except ImportError:
        pass

if not IS_TPU:
    if torch.cuda.is_available():
        gpu_count = torch.cuda.device_count()
        gpu_name = torch.cuda.get_device_name(0)
        DEVICE_STR = f"CUDA ({gpu_count}x {gpu_name})"
        print(f"[HARDWARE] SUCCESS: Detected {DEVICE_STR}")
    else:
        print("[HARDWARE] Running on CPU / Standard environment")

print(f"[FRAMEWORK] PyTorch: {torch.__version__}")

## 4. Secrets Diagnostics, Hub Auth & SFT Reference Model Setup

Audits environment variables from `.env`, Kaggle Secrets Vault, and Colab Userdata, authenticates with Hugging Face Hub, and verifies the Stage 1 SFT reference checkpoint.

In [ ]:
# =========================================================================
# 4. PANORAMIC RADIOGRAPHIC IMAGES INGESTION (DUAL-PATH ARCHITECTURE)
# =========================================================================
# Path A (Attached Kaggle Dataset): Auto-detects /kaggle/input/dentex-panoramic
#        and /kaggle/input/tufts-panoramic if attached to this session.
#        (0 download time, 0 MB disk space used from your 40 GB workspace quota).
# Path B (Traditional Hugging Face Hub): Downloads directly from Hugging Face Hub
#        ('Reza-Nadimi/dentex-train-images' and 'Reza-Nadimi/tufts-train-images').
#
# TOGGLE: Set FORCE_HF_DATASET_DOWNLOAD = True to force remote Hugging Face
# download even when Kaggle inputs are attached.
FORCE_HF_DATASET_DOWNLOAD = False

# --- A. DENTEX Ingestion ---
dentex_local = Path("data/dentex")
dentex_local.mkdir(parents=True, exist_ok=True)
kaggle_dentex_candidates = [
    Path("/kaggle/input/datasets/rezanadimikj/dentex-panoramic"),
    Path("/kaggle/input/dentex-panoramic"),
    Path("/kaggle/input/dentex"),
]
dentex_mounted = False

if not FORCE_HF_DATASET_DOWNLOAD:
    for cand in kaggle_dentex_candidates:
        if cand.exists():
            target_img_dir = cand / "images" if (cand / "images").exists() else cand
            link_target = dentex_local / "images"
            if not link_target.exists():
                try:
                    link_target.symlink_to(target_img_dir)
                except Exception:
                    pass
            print(f"[DATASET] Attached Kaggle DENTEX detected at {cand} (Skipping remote download).")
            dentex_mounted = True
            break

if not dentex_mounted:
    dentex_repo = os.environ.get("DENTEX_IMAGES_REPO", "Reza-Nadimi/dentex-train-images")
    has_dentex = (dentex_local / "images").exists() or (dentex_local / "train_images").exists() or Path("data/images").exists()
    if dentex_repo and not has_dentex:
        print(f"\n[SYNC] Downloading DENTEX panoramic images from Hugging Face ({dentex_repo})...")
        try:
            snapshot_download(
                repo_id=dentex_repo,
                repo_type="dataset",
                local_dir=str(dentex_local),
                token=hf_token,
            )
            print("[SYNC] DENTEX images ready.")
        except Exception as e:
            print(f"[SYNC WARNING] DENTEX images download failed: {e}")
    else:
        print("\n[SYNC] DENTEX panoramic images verified on local disk.")

# --- B. Tufts Ingestion ---
tufts_local = Path("data/tufts")
tufts_local.mkdir(parents=True, exist_ok=True)
kaggle_tufts_candidates = [
    Path("/kaggle/input/datasets/rezanadimikj/tufts-panoramic"),
    Path("/kaggle/input/tufts-panoramic"),
    Path("/kaggle/input/tufts"),
]
tufts_mounted = False

if not FORCE_HF_DATASET_DOWNLOAD:
    for cand in kaggle_tufts_candidates:
        if cand.exists():
            target_img_dir = cand / "Radiographs" if (cand / "Radiographs").exists() else cand
            link_target = tufts_local / "Radiographs"
            if not link_target.exists():
                try:
                    link_target.symlink_to(target_img_dir)
                except Exception:
                    pass
            print(f"[DATASET] Attached Kaggle Tufts detected at {cand} (Skipping remote download).")
            tufts_mounted = True
            break

if not tufts_mounted:
    tufts_repo = os.environ.get("TUFTS_IMAGES_REPO", "Reza-Nadimi/tufts-train-images")
    has_tufts = (tufts_local / "Radiographs").exists() or (tufts_local / "images").exists() or Path("data/Tufts/Radiographs").exists()
    if tufts_repo and not has_tufts:
        print(f"\n[SYNC] Downloading Tufts panoramic images from Hugging Face ({tufts_repo})...")
        try:
            snapshot_download(
                repo_id=tufts_repo,
                repo_type="dataset",
                local_dir=str(tufts_local),
                token=hf_token,
            )
            print("[SYNC] Tufts images ready.")
        except Exception as e:
            print(f"[SYNC WARNING] Tufts images download failed: {e}")
    else:
        print("[SYNC] Tufts panoramic images verified on local disk.")


## 5. [G3] Dual-LoRA Reference/Policy Toggle Verification Test

Runs mathematical unit tests verifying adapter toggling between frozen reference and trainable policy, advantage normalization, and non-negative KL divergence.

In [ ]:
print("[VERIFY] Executing G3 Dual-LoRA Adapter Toggle and Advantage Normalization Tests...")
!python -m pytest tests/test_dual_adapter_grpo.py tests/test_grpo_advantages.py -v

## 6. Interactive GRPO Configuration & Execution Manifest



In [ ]:
from tabulate import tabulate

# =========================================================================
# STAGE 2 GRPO EXECUTION PARAMETERS
# =========================================================================
# Training Track:
#   - 'with_tools' : Multi-turn diagnostic agent with 8 workstation tools
#   - 'no_tools'   : Direct radiologist (tool-free Chain-of-Thought)
# =========================================================================
# BASE MODEL RESOLUTION: Attached Kaggle Input (Default) vs. Hugging Face Hub
# =========================================================================
# Path A (Default on Kaggle): Loads directly from attached private Kaggle input:
#   /kaggle/input/qwen3-5-9b (rezanadimikj/qwen3-5-9b)
#   -> 0 MB downloaded across the internet
#   -> 0 MB disk consumed from your 40.8 GB workspace quota
# Path B (Traditional / Auditor Fallback): Downloads directly from Hugging Face Hub:
#   'Qwen/Qwen3.5-9B'
#
# TOGGLE: Set FORCE_HF_MODEL_DOWNLOAD = True to force downloading from Hugging Face.
FORCE_HF_MODEL_DOWNLOAD = False

def resolve_base_model(default_repo="Qwen/Qwen3.5-9B") -> str:
    if not FORCE_HF_MODEL_DOWNLOAD:
        candidates = [
            Path("/kaggle/input/datasets/rezanadimikj/qwen3-5-9b"),
            Path("/kaggle/input/qwen3-5-9b"),
            Path("/kaggle/input/qwen3_5_9b"),
            Path("/kaggle/input/qwen-3-5-9b"),
            Path("/kaggle/input/qwen3-5-9b-base"),
        ]
        for cand in candidates:
            if cand.is_dir() and (cand / "config.json").exists():
                print(f"[RESOLVER] Detected attached Kaggle Base Model: {cand}")
                print("           (0 download, 0 disk space used from 40 GB workspace quota)")
                return str(cand)
    print(f"[RESOLVER] Using Hugging Face Hub model: {default_repo}")
    return default_repo

MODEL_ID = resolve_base_model()

TRACK = "with_tools"

# Curriculum SFT Reference Stage:
#   - 'dentex_alone'         : Stage 1a baseline reference adapter
#   - 'dentex_tufts_overlap' : Stage 1b overlapping diseases reference adapter
#   - 'multicohort_all'      : Stage 1c full multi-cohort reference adapter
SFT_STAGE = "dentex_alone"

# Execution Mode:
#   - 'single' : Run optimization for a single group size K
#   - 'sweep'  : Automatically sweep across K in [1, 2, 4, 8, 16]
MODE = "single"

# Group Size K (for 'single' mode: 1, 2, 4, 8, or 16)
GROUP_SIZE = 4
K_VALUES = [1, 2, 4, 8, 16]

# Hardware & Multi-Core Distribution (Cloud TPU v5e-8)
NUM_CORES = 8 if IS_TPU else 1
USE_FSDP = True if IS_TPU else False

# RL Optimization Hyperparameters
EPOCHS = 2
LEARNING_RATE = 5e-6
KL_BETA = 0.04
CLIP_EPS = 0.2
DATASET = "dentex"

# Checkpoint Sync & Kaggle Continuity
HF_REPO = os.environ.get("HF_ARTIFACT_REPO", "Reza-Nadimi/vlm-dental-models")
PUSH_EVERY_STEPS = 25
RESUME = False

# Target Checkpoint & Remote Hub Folder
target_name = f"qwen3_5_9b_grpo_{TRACK}_k{GROUP_SIZE}_{SFT_STAGE}"
target_checkpoint = f"data/models/{target_name}"
hf_target_folder = f"grpo/{target_name}"
sft_ref_folder = f"sft/qwen3_5_9b_sft_{TRACK}_{SFT_STAGE}"

manifest_data = [
    ["Execution Mode", MODE.upper()],
    ["Training Track", TRACK],
    ["Base Model ID", MODEL_ID],
    ["SFT Reference Stage", SFT_STAGE],
    ["SFT Ref Subfolder", sft_ref_folder],
    ["Group Size (K)", str(GROUP_SIZE) if MODE == "single" else str(K_VALUES)],
    ["Dataset", DATASET.upper()],
    ["Policy Learning Rate", f"{LEARNING_RATE}"],
    ["Schulman k3 KL Beta", f"{KL_BETA}"],
    ["PPO Clipping Epsilon", f"{CLIP_EPS}"],
    ["Epochs per Batch", str(EPOCHS)],
    ["Target Local Checkpoint", target_checkpoint],
    ["HF Models Repository", HF_REPO if HF_REPO else "Disabled"],
    ["HF Checkpoint Subfolder", hf_target_folder],
    ["TPU Multi-Core Topology", f"{NUM_CORES} cores (xmp.spawn)" if IS_TPU else "Single device (1 core/GPU)"],
    ["FSDP Parameter Sharding", f"Enabled (~2.3 GB base weights / core)" if IS_TPU and USE_FSDP else "Disabled (DDP / Single)"],
    ["Resume from HF", str(RESUME)],
]

print("=" * 75)
print("VLM-DENTAL: STAGE 2 GRPO EXECUTION MANIFEST")
print("=" * 75)
print(tabulate(manifest_data, headers=["Configuration Parameter", "Assigned Value"], tablefmt="fancy_grid"))

## 7. Launch Stage 2 GRPO Policy Optimization Pipeline



In [ ]:
if MODE == "sweep":
    print(f"[LAUNCHING SWEEP] Executing GRPO K in {K_VALUES} sweep orchestrator...")
    cmd = [
        "python", "scripts/run_grpo_sweep.py",
        "--track", TRACK,
        "--model-id", MODEL_ID,
        "--sft-stage", SFT_STAGE,
        "--k-values", *[str(k) for k in K_VALUES],
        "--epochs", str(EPOCHS),
        "--lr", str(LEARNING_RATE),
        "--hf-repo", HF_REPO,
        "--num-cores", str(NUM_CORES),
        "--fsdp" if USE_FSDP else "--no-fsdp",
    ]
else:
    print(f"[LAUNCHING GRPO] Executing GRPO training with K={GROUP_SIZE} on {TRACK}...")
    cmd = [
        "python", "scripts/run_grpo.py",
        "--track", TRACK,
        "--model-id", MODEL_ID,
        "--sft-stage", SFT_STAGE,
        "--dataset", DATASET,
        "--group-size", str(GROUP_SIZE),
        "--epochs", str(EPOCHS),
        "--lr", str(LEARNING_RATE),
        "--kl-beta", str(KL_BETA),
        "--clip-eps", str(CLIP_EPS),
        "--hf-repo", HF_REPO,
        "--push-every-steps", str(PUSH_EVERY_STEPS),
        "--num-cores", str(NUM_CORES),
        "--fsdp" if USE_FSDP else "--no-fsdp",
    ]
    if RESUME:
        cmd.extend(["--resume-hf", HF_REPO])

# Clear conflicting legacy TPU multi-host cluster address variables on Kaggle/Colab
for _var in ["TPU_PROCESS_ADDRESSES", "TPU_PROCESS_COUNT", "CLOUD_TPU_TASK_ID", "PJRT_DEVICE"]:
    os.environ.pop(_var, None)

cmd_str = " ".join(cmd)
print(f"[PIPELINE COMMAND] Executing:\n{cmd_str}\n")
# Clean up any lingering TPU device processes before launching
!pkill -9 -f "multiprocessing.spawn" 2>/dev/null || true
!pkill -9 -f "multiprocessing.resource_tracker" 2>/dev/null || true
!pkill -9 -f "train_sft.py" 2>/dev/null || true
!pkill -9 -f "run_grpo.py" 2>/dev/null || true
!fuser -k -9 /dev/vfio/* 2>/dev/null || true
!{cmd_str}

## 8. Real-Time Dual-Axis Reward & KL Convergence Dashboard

Visualizes reinforcement learning dynamics across training steps, plotting Mean Trajectory Reward on the primary axis (left, green) and Schulman k3 KL Divergence on the secondary axis (right, red dashed).

In [ ]:
import json
import matplotlib.pyplot as plt

log_candidates = [
    f"{target_checkpoint}/grpo_training_log.jsonl",
    "data/grpo_training_log.jsonl",
    "data/eval_results/grpo_training_log.jsonl",
]

log_file = None
for cand in log_candidates:
    if os.path.exists(cand):
        log_file = cand
        break

if log_file:
    steps, rewards, kls = [], [], []
    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                r = json.loads(line)
                steps.append(r.get("step", len(steps) + 1))
                rewards.append(r.get("mean_reward", 0.0))
                kls.append(r.get("kl_divergence", 0.0))

    fig, ax1 = plt.subplots(figsize=(11, 5))

    color_reward = "#2ca02c"
    ax1.set_xlabel("GRPO Rollout Steps", fontsize=11)
    ax1.set_ylabel("Mean Trajectory Reward", color=color_reward, fontsize=11)
    ax1.plot(steps, rewards, color=color_reward, lw=2.0, marker="o", markersize=4, label="Mean Reward")
    ax1.tick_params(axis="y", labelcolor=color_reward)
    ax1.grid(True, alpha=0.3)

    if any(k > 0 for k in kls):
        ax2 = ax1.twinx()
        color_kl = "#d62728"
        ax2.set_ylabel("KL Divergence (Reference vs Policy)", color=color_kl, fontsize=11)
        ax2.plot(steps, kls, color=color_kl, lw=1.8, linestyle="--", label="KL Divergence")
        ax2.tick_params(axis="y", labelcolor=color_kl)

    plt.title(f"VLM-DENTAL Stage 2 GRPO Convergence: {TRACK.upper()} (K={GROUP_SIZE} | SFT: {SFT_STAGE})", fontsize=12, fontweight="bold")
    fig.tight_layout()
    plt.show()
    print(f"[METRICS] Latest Rollout Reward: {rewards[-1]:.4f} | Latest KL: {kls[-1]:.4f}")
else:
    print(f"[INFO] Log file not generated yet. Execute Cell 7 to start GRPO training.")